In [2]:
import os
import numpy as np
import torch
import joblib
import gradio as gr

from PIL import Image

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

print("CUDA available:", torch.cuda.is_available())
print("CUDA version used by PyTorch:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version used by PyTorch: 12.6
GPU: NVIDIA GeForce RTX 3070


In [3]:
DATASET_PATH = "image_dataset/train"
BATCH_SIZE = 32

MODEL_SAVE_PATH = "svm_classifier.joblib"
CLASS_NAMES_SAVE_PATH = "class_names.joblib"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

class_names = dataset.classes

print("Class names:", class_names)
print("Class mapping:", dataset.class_to_idx)
print("Number of images:", len(dataset))

Class names: ['A Lull in the Sea', 'A Place Further Than The Universe', 'A Silent Voice', 'AKIRA', 'Ace of Diamond', 'Akatsuki no Yona', 'Angel Beats!', 'Aria The Natural', 'Aria The Origination', 'Assassination Classroom', 'Attack on Titan', 'Baccano!', 'Bakemonogatari', 'Bakuman', 'Barakamon', 'Beck Mongolian Chop Squad', 'Berserk', 'Big Windup!', 'Black Butler', 'Black Lagoon', 'Bunny Drop', 'Cardcaptor Sakura', 'Carpcaptor Sakura', 'Castle in the Sky', 'Chihayafuru', 'Clannad', 'Code Geass', 'Cross Game', 'D.Gray-man', 'DARLING in the FRANXX', 'Daily Lives of High School Boys', 'Darker than Black', 'Death Note', 'Death Parade', 'Den-noh Coil', 'Descending Stories Showa Genroku Rakugo Shinju', 'Detective Conan', 'Detroit Metal City The Animated Series', 'Dragon Ball', 'Dragon Ball Z', 'Durarara', 'Durarara!!', 'ERASED', 'Erin', 'Eureka Seven', 'FateZero', 'Fatestay night', 'Fighting Spirit Special', 'From the New World', 'Full Metal Panic', 'Fullmetal Alchemist Brotherhood', 'Gankut

In [6]:
indices = np.arange(len(dataset))
labels = np.array([label for _, label in dataset.samples])

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.1,
    random_state=42,
    stratify=labels
)

train_subset = Subset(dataset, train_idx)
test_subset = Subset(dataset, test_idx)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

print("Training images:", len(train_subset))
print("Testing images:", len(test_subset))

Training images: 74677
Testing images: 8298


In [7]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Remove the final classification layer
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])

feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [8]:
def extract_features(loader):
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            features = feature_extractor(images)

            # [batch_size, 512, 1, 1] → [batch_size, 512]
            features = features.view(features.size(0), -1)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    X = np.vstack(all_features)
    y = np.concatenate(all_labels)

    return X, y

In [11]:
feature_files_exist = (
    os.path.exists("X_train.npy") and
    os.path.exists("y_train.npy") and
    os.path.exists("X_test.npy") and
    os.path.exists("y_test.npy")
)

if feature_files_exist:
    print("Saved features found. Loading from disk...")

    X_train = np.load("X_train.npy")
    y_train = np.load("y_train.npy")
    X_test = np.load("X_test.npy")
    y_test = np.load("y_test.npy")

else:
    print("Saved features not found. Extracting CNN features...")

    X_train, y_train = extract_features(train_loader)
    X_test, y_test = extract_features(test_loader)

    np.save("X_train.npy", X_train)
    np.save("y_train.npy", y_train)
    np.save("X_test.npy", X_test)
    np.save("y_test.npy", y_test)

    print("Features extracted and saved.")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Saved features found. Loading from disk...
X_train shape: (74677, 512)
X_test shape: (8298, 512)


In [13]:
from sklearn.linear_model import LogisticRegression

MODEL_PATH = "logistic_regression_model.joblib"

if os.path.exists(MODEL_PATH):
    print("Saved model found. Loading model...")
    model = joblib.load(MODEL_PATH)

else:
    print("No saved model found. Training model...")

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    joblib.dump(model, MODEL_PATH)
    print("Model trained and saved.")

print(type(model))

No saved model found. Training model...
Model trained and saved.
<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [14]:
y_pred = model.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=class_names
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

                                                               precision    recall  f1-score   support

                                            A Lull in the Sea       0.31      0.32      0.31        38
                            A Place Further Than The Universe       0.45      0.49      0.47        37
                                               A Silent Voice       0.31      0.30      0.30        27
                                                        AKIRA       0.34      0.26      0.30        38
                                               Ace of Diamond       0.41      0.42      0.42        38
                                             Akatsuki no Yona       0.17      0.11      0.13        38
                                                 Angel Beats!       0.33      0.47      0.39        38
                                             Aria The Natural       0.30      0.21      0.25        38
                                         Aria The Origination       0.33

In [15]:
joblib.dump(LogisticRegression, MODEL_SAVE_PATH)
joblib.dump(class_names, CLASS_NAMES_SAVE_PATH)

print("Saved model to:", MODEL_SAVE_PATH)
print("Saved class names to:", CLASS_NAMES_SAVE_PATH)

Saved model to: svm_classifier.joblib
Saved class names to: class_names.joblib


In [16]:
def image_to_cnn_features(pil_image):
    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")

    image_tensor = transform(pil_image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = feature_extractor(image_tensor)
        features = features.view(features.size(0), -1)

    return features.cpu().numpy()

In [17]:
def predict_class(image):
    if image is None:
        return {"No image uploaded": 1.0}

    pil_image = Image.fromarray(image.astype("uint8"))

    features = image_to_cnn_features(pil_image)

    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]

    results = {
        class_names[i]: float(probabilities[i])
        for i in range(len(class_names))
    }

    return results

In [18]:
interface = gr.Interface(
    fn=predict_class,
    inputs=gr.Image(type="numpy", label="Upload image"),
    outputs=gr.Label(num_top_classes=3, label="Predicted Class"),
    title="Image Classifier",
    description="Upload an image."
)

interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://5adb6a60b2ea05661c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
